# Dataset Exploration

**Objective:** Analyze the FaceForensics++ and Celeb-DF datasets to understand structure, distribution, and characteristics.

**Research Traceability:**
- Research Objective: Dataset acquisition and exploration
- Methodology: Statistical analysis of video metadata
- Implementation: notebooks/01_dataset_exploration.ipynb

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Dataset Structure

In [ ]:
# Define paths
DATA_DIR = Path('../datasets/raw')

# Check available datasets
datasets = {}
for dataset_name in ['faceforensics', 'celebdf']:
    dataset_path = DATA_DIR / dataset_name
    if dataset_path.exists():
        datasets[dataset_name] = dataset_path
        print(f"[✓] {dataset_name}: {dataset_path}")
    else:
        print(f"[✗] {dataset_name}: {dataset_path} (not found)")

print(f"\nAvailable datasets: {list(datasets.keys())}")

## 2. FaceForensics++ Analysis

In [ ]:
def analyze_faceforensics(base_path: Path) -> pd.DataFrame:
    """Analyze FaceForensics++ dataset structure."""
    records = []
    
    # Manipulation methods
    methods = ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']
    
    for method in methods:
        method_path = base_path / 'manipulation' / method
        if method_path.exists():
            videos = list(method_path.rglob('*.mp4'))
            for video in videos[:5]:  # Sample first 5
                records.append({
                    'dataset': 'FaceForensics++',
                    'method': method,
                    'label': 'fake',
                    'path': str(video),
                    'filename': video.name,
                })
    
    # Original videos
    original_path = base_path / 'original'
    if original_path.exists():
        videos = list(original_path.rglob('*.mp4'))
        for video in videos[:5]:
            records.append({
                'dataset': 'FaceForensics++',
                'method': 'original',
                'label': 'real',
                'path': str(video),
                'filename': video.name,
            })
    
    return pd.DataFrame(records)

if 'faceforensics' in datasets:
    ff_df = analyze_faceforensics(datasets['faceforensics'])
    print(f"FaceForensics++ samples: {len(ff_df)}")
    print(ff_df.head())
else:
    print("FaceForensics++ dataset not available")

## 3. Celeb-DF Analysis

In [ ]:
def analyze_celebdf(base_path: Path) -> pd.DataFrame:
    """Analyze Celeb-DF dataset structure."""
    records = []
    
    categories = {
        'Celeb-real': 'real',
        'Celeb-synthesis': 'fake',
        'YouTube-real': 'real',
    }
    
    for category, label in categories.items():
        cat_path = base_path / category
        if cat_path.exists():
            videos = list(cat_path.rglob('*.mp4'))
            for video in videos[:5]:
                records.append({
                    'dataset': 'Celeb-DF',
                    'method': category,
                    'label': label,
                    'path': str(video),
                    'filename': video.name,
                })
    
    return pd.DataFrame(records)

if 'celebdf' in datasets:
    celeb_df = analyze_celebdf(datasets['celebdf'])
    print(f"Celeb-DF samples: {len(celeb_df)}")
    print(celeb_df.head())
else:
    print("Celeb-DF dataset not available")

## 4. Dataset Statistics

In [ ]:
# Combine all data
all_data = pd.concat([ff_df, celeb_df], ignore_index=True)

# Summary statistics
print("=" * 60)
print("Dataset Summary")
print("=" * 60)
print(f"Total samples: {len(all_data)}")
print(f"\nBy label:")
print(all_data['label'].value_counts())
print(f"\nBy method:")
print(all_data['method'].value_counts())

In [ ]:
# Visualize distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Label distribution
all_data['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Label Distribution')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')

# Method distribution
all_data['method'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Manipulation Method Distribution')
axes[1].set_xlabel('Method')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/plots/dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Video Quality Analysis

In [ ]:
import cv2

def get_video_info(video_path: str) -> dict:
    """Extract video metadata."""
    cap = cv2.VideoCapture(video_path)
    info = {
        'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        'fps': cap.get(cv2.CAP_PROP_FPS),
        'frame_count': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    info['duration'] = info['frame_count'] / info['fps'] if info['fps'] > 0 else 0
    cap.release()
    return info

# Analyze sample videos
sample_videos = all_data.head(10)
video_infos = []

for _, row in sample_videos.iterrows():
    try:
        info = get_video_info(row['path'])
        info['filename'] = row['filename']
        info['label'] = row['label']
        video_infos.append(info)
    except Exception as e:
        print(f"Error processing {row['filename']}: {e}")

video_df = pd.DataFrame(video_infos)
print("Video Statistics:")
print(video_df.describe())

## 6. Key Findings

1. **Dataset Structure**: FaceForensics++ contains 4 manipulation methods + originals; Celeb-DF has real/synthesized celebrity faces.
2. **Class Balance**: Check if real/fake distribution is balanced across datasets.
3. **Video Quality**: Resolution and FPS vary across sources.
4. **Manipulation Artifacts**: Different methods produce different visual artifacts.

## Next Steps
- Proceed to preprocessing: `02_preprocessing.ipynb`
- Face detection and extraction: `03_face_detection.ipynb`